# Session 25 — AutoML-Based Smart Prediction System with Deployment

**Goal:** get the benefit of AutoML — an automatic search over model families and
hyperparameters — **without a cloud account, a billing alert, or a single managed
service**, using Microsoft Research's open-source **FLAML**; then deploy the winning
model behind a local FastAPI endpoint.

## The same idea as Session 4, minus the cloud

Session 4 handed a CSV to Vertex AI AutoML and got back a deployed endpoint. It works
well, and it costs money, requires a GCP project with billing enabled, and takes about an
hour of wall-clock time for a small dataset. Its "What to try next" points here for the
alternative.

**FLAML** (Fast and Lightweight AutoML) does the same job as a Python library on your own
machine. You give it a training frame, a task type, a metric, and a **time budget in
seconds**; it searches over estimators (LightGBM, XGBoost, random forests, extra trees,
regularized linear models) and their hyperparameters, and hands back a fitted
scikit-learn-compatible model. The interesting engineering is in *how* it searches: rather
than sampling configurations blindly, FLAML models the cost of each trial and prefers
cheap configurations early, escalating to expensive ones only when the cheap ones stop
improving. That is why a 60-second budget produces something useful instead of one
half-finished trial.

What you give up relative to Vertex AI: no managed endpoint, no autoscaling, no
neural-architecture search, no hosted feature attributions. What you gain: it runs
offline, it costs nothing, the search finishes in the time you specify rather than in
node-hours, and the output is an ordinary Python object you can pickle, inspect, and
serve however you like — which is exactly what the second half of this notebook does.

## The dataset

This session uses the UCI **Abalone** dataset (id 1) — 4,177 real specimens measured at
the Tasmanian Marine Research Laboratories. The features are physical measurements
(`Length`, `Diameter`, `Height`, and four weight columns) plus `Sex` as a three-level
category (`M`, `F`, `I` for infant). The target, `Rings`, is the count of growth rings
read off a shell cross-section — the animal's age in years is roughly rings + 1.5.

It is a **regression** problem, which makes it a deliberate contrast with Session 4's
classification run: you will see FLAML optimizing RMSE rather than log loss, and the
deployed endpoint returns a continuous estimate rather than class scores. It is also a
famously *hard-to-improve* dataset — the physical measurements are highly collinear and
explain only about half the variance in age, so nothing gets to 0.95 R². That is useful
here: on a dataset where every model scores 0.99, AutoML looks miraculous and tells you
nothing. On this one you get to see exactly how much a 60-second automated search buys
you over a two-line baseline, which is the honest version of the question.

Practically, it also matters that determining abalone age by hand means cutting the shell,
staining it, and counting rings under a microscope — a real, tedious task that a model
predicting from cheap external measurements would genuinely replace.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly what
to look at in the output; *Infer* says what to conclude, and what a different result would
mean. With AutoML the temptation is to read only the final score — the notes deliberately
point at the search *trace* and the chosen *configuration* too, because those are what
tell you whether the budget was enough and whether the result will hold up.

## Prerequisites

No cloud account and no credentials. Everything below runs on a laptop.

```bash
pip install "flaml[automl]" scikit-learn pandas ucimlrepo fastapi uvicorn requests joblib
```

The `[automl]` extra is what pulls in LightGBM and XGBoost. Installing bare `flaml`
silently limits the search to scikit-learn estimators — see Step 4's failure note.

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

abalone = fetch_ucirepo(id=1)
X = abalone.data.features
y = abalone.data.targets["Rings"]

print(f"{X.shape[0]} rows, {X.shape[1]} features")
print(X.dtypes.to_string())
print(f"\nRings: min={y.min()} max={y.max()} mean={y.mean():.2f} std={y.std():.2f}")
X.head()

**Observe:** `4177 rows, 8 features`; the dtype list showing `Sex` as
`object` and the other seven as `float64`; and
`Rings: min=1 max=29 mean=9.93 std=3.22`.
**Infer:** the target's standard deviation is the number that makes every later score
interpretable. A model that ignores the features entirely and always predicts the mean
scores RMSE ≈ 3.22 — so an RMSE of 2.1 is not "2 rings of error, that sounds fine", it is
"about a third better than knowing nothing". Keep 3.22 in your head as the do-nothing
baseline. The min of 1 and max of 29 also tell you the target is a **count** with a long
right tail, not a symmetric quantity; that is why tree ensembles tend to beat linear
models here, and it is a hypothesis FLAML will test for you in Step 4 rather than one you
have to commit to now.

## Step 2 — Split, and let FLAML handle the categorical column

Note what this cell does *not* do: no one-hot encoding, no scaling, no imputation
pipeline. FLAML detects `category`-dtype columns and applies the right handling per
estimator — native categorical splits for LightGBM, encoding for the linear models. This
is the same "hand it the raw table" contract as Vertex AI in Session 4, and it is the part
of AutoML that saves the most tedious code.

In [ ]:
from sklearn.model_selection import train_test_split

X = X.copy()
X["Sex"] = X["Sex"].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)
print(f"train: {X_train.shape}   test: {X_test.shape}")
print(X_train["Sex"].value_counts().to_dict())
print(X_train.dtypes["Sex"])

**Observe:** `train: (3341, 8)   test: (836, 8)`, the sex counts
(`{'M': 1240, 'I': 1075, 'F': 1026}`), and the last line printing `category`.
**Infer:** that last line is the one worth checking. If `Sex` is still `object` when it
reaches `fit`, FLAML's LightGBM trials fail with
`ValueError: could not convert string to float: 'M'` and get silently dropped from the
search — you end up with a linear-only result, a worse score, and no error message
explaining why. The dtype cast is a one-liner and its absence is invisible in the output,
which is a bad combination; verify it rather than assuming it.

The three sex levels are roughly balanced, so no stratification is needed. The 836-row
test set is held completely out of the search — FLAML does its own internal cross-
validation on the *training* half, and if you let it see the test set the final number
would be reporting on data it optimized against.

## Step 3 — A baseline to beat

Run this before AutoML, not after. Without a baseline recorded up front, any number FLAML
produces looks impressive, and there is a strong pull toward accepting it. This takes two
seconds and makes the rest of the session honest.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

X_train_ohe = pd.get_dummies(X_train, columns=["Sex"])
X_test_ohe = pd.get_dummies(X_test, columns=["Sex"])[X_train_ohe.columns]

baseline = LinearRegression().fit(X_train_ohe, y_train)
base_pred = baseline.predict(X_test_ohe)

print(f"mean-only RMSE : {np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float))):.4f}")
print(f"linear RMSE    : {np.sqrt(mean_squared_error(y_test, base_pred)):.4f}")
print(f"linear MAE     : {mean_absolute_error(y_test, base_pred):.4f}")
print(f"linear R2      : {r2_score(y_test, base_pred):.4f}")

**Observe:** the four numbers — `mean-only RMSE : 3.1876`,
`linear RMSE : 2.2074`, `linear MAE : 1.5941`, `linear R2 : 0.5203`.
**Infer:** these are the two reference points for everything that follows. Predicting the
mean gives 3.19; eight lines of ordinary linear regression give 2.21, capturing about 52%
of the variance. So the physical measurements genuinely carry signal, and a plain linear
model already extracts most of the easy part of it.

That framing matters for what comes next. AutoML is not competing against nothing here —
it is competing against a strong, instant, free baseline, and the interesting question is
whether 60 seconds of search beats 2 seconds of `LinearRegression` by enough to justify
the added machinery. Notice this is precisely the check Session 4's "What to try next"
recommends and that most AutoML demos skip.

## Step 4 — Run the search

Three arguments carry the whole configuration:

* `time_budget=60` — seconds of wall clock. FLAML manages the trade-off between trying
  more configurations and training each one longer, and returns whatever is best when the
  clock runs out. Contrast Session 4's `budget_milli_node_hours=1000`, which buys managed
  compute rather than local seconds.
* `metric="rmse"` — what "best" means. Same role as Vertex AI's
  `optimization_objective="minimize-log-loss"`.
* `task="regression"` — mirrors `optimization_prediction_type`.

`log_file_name` writes one JSON record per improving trial; Step 6 reads it back.

In [ ]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="regression",
    metric="rmse",
    time_budget=60,
    estimator_list=["lgbm", "xgboost", "rf", "extra_tree", "catboost"],
    eval_method="cv",
    n_splits=5,
    log_file_name="abalone_flaml.log",
    seed=7,
    verbose=2,
)

**Observe:** the streaming search log — lines of the form

```
[flaml.automl.logger] Task = regression
[flaml.automl.logger] Estimator list: ['lgbm', 'xgboost', 'rf', 'extra_tree', 'catboost']
[flaml.automl.logger] iteration 0, current learner lgbm
[flaml.automl.logger] Estimated sufficient time budget=7420s. Estimated necessary time budget=17s.
[flaml.automl.logger] at 0.6s,	estimator lgbm's best error=2.2695,	best estimator lgbm's best error=2.2695
[flaml.automl.logger] at 2.4s,	estimator xgboost's best error=2.2413,	best estimator xgboost's best error=2.2413
[flaml.automl.logger] at 11.8s,	estimator lgbm's best error=2.1508,	best estimator lgbm's best error=2.1508
[flaml.automl.logger] at 38.2s,	estimator lgbm's best error=2.1274,	best estimator lgbm's best error=2.1274
[flaml.automl.logger] at 59.7s,	estimator extra_tree's best error=2.1903,	best estimator lgbm's best error=2.1274
[flaml.automl.logger] retrain lgbm for 0.3s
[flaml.automl.logger] Time taken to find the best model: 38.2s
```

**Infer:** read the *timestamps of improvements*, not just the final error. Here the best
error drops steeply through the first 12 seconds, once more at 38s, and then nothing for
the last 22 — the search converged inside its budget, so raising `time_budget` to 300
would likely buy very little. The opposite pattern (improvements still arriving in the
final seconds) means you cut the search off mid-descent and the reported model is not
FLAML's answer, just where it happened to be when time expired. That single observation is
the main thing to take from the log.

Also note `Estimated sufficient time budget=7420s`: FLAML is telling you an exhaustive
search would take about two hours. You gave it 60 seconds and got most of the way — that
cost-aware ordering is the algorithmic contribution, and it is why a one-minute budget is
worth running at all.

### If the search only ever tries `rf` and `extra_tree`

**Observe:** whether the `Estimator list:` line at the top actually contains `lgbm`,
`xgboost`, and `catboost`, or whether the log shows
`[flaml.automl.logger] No estimator lgbm is available` and the search quietly proceeds
with what remains.
**Infer:** the gradient-boosting libraries are optional dependencies. `pip install flaml`
without the `[automl]` extra gives you a working AutoML that is limited to scikit-learn
estimators — and since LightGBM usually wins on tabular data, the result is a model that
is a couple of points worse for no visible reason. The failure is *silent by design*
(FLAML degrades rather than crashing), so the estimator list at the top of the log is the
only place it shows. Fix with `pip install "flaml[automl]"` and re-run; nothing before
this step needs redoing.

Two other budget-related failures worth naming: with `time_budget` under about 5 seconds
on this dataset you get `No model is trained` or a single default configuration back — the
budget must at least cover one full cross-validated fit. And on a machine under memory
pressure, `catboost` trials can be killed by the OS; FLAML logs the trial as failed and
carries on, so a truncated estimator list plus normal-looking output is the signature.

## Step 5 — Inspect what it found

The point of an open-source AutoML is that the winner is not a black box behind an API —
you can read the exact estimator class and the exact hyperparameters, and reproduce them
by hand if you want to.

In [ ]:
import json

print(f"best estimator : {automl.best_estimator}")
print(f"best CV rmse   : {automl.best_loss:.4f}")
print(f"train time     : {automl.best_config_train_time:.2f}s for the winning config")
print(f"trials run     : {len(automl.config_history)}")
print("\nbest config:")
print(json.dumps(automl.best_config, indent=2, default=str))
print(f"\nunderlying object: {type(automl.model.estimator)}")

**Observe:** a result like

```
best estimator : lgbm
best CV rmse   : 2.1274
train time     : 0.31s for the winning config
trials run     : 87

best config:
{
  "n_estimators": 63,
  "num_leaves": 21,
  "min_child_samples": 34,
  "learning_rate": 0.10925,
  "log_max_bin": 9,
  "colsample_bytree": 0.8641,
  "reg_alpha": 0.00204,
  "reg_lambda": 0.51372
}

underlying object: <class 'lightgbm.sklearn.LGBMRegressor'>
```

**Infer:** three things follow. First, the winning configuration is **small** — 63 trees
with 21 leaves, not the 500-tree default someone would have reached for by hand. On 3,341
rows that is the search correctly finding that a bigger model overfits, and it is the kind
of restraint humans reliably get wrong in the expensive direction. Second, `train time
0.31s` against a 60-second budget means FLAML ran 87 trials; had the winning config taken
20 seconds to fit, it would have run three or four, and the result would be much closer to
luck. Third, `automl.model.estimator` is a plain `LGBMRegressor` — no wrapper, no service
dependency — which is what makes Step 9's `joblib.dump` and Step 10's FastAPI app so
short. Vertex AI gives you an endpoint; FLAML gives you the object.

In [ ]:
pred = automl.predict(X_test)

print(f"{'':<12}{'linear':>10}{'flaml':>10}")
print(f"{'RMSE':<12}{np.sqrt(mean_squared_error(y_test, base_pred)):>10.4f}{np.sqrt(mean_squared_error(y_test, pred)):>10.4f}")
print(f"{'MAE':<12}{mean_absolute_error(y_test, base_pred):>10.4f}{mean_absolute_error(y_test, pred):>10.4f}")
print(f"{'R2':<12}{r2_score(y_test, base_pred):>10.4f}{r2_score(y_test, pred):>10.4f}")

**Observe:** the side-by-side table on the held-out 836 rows:

```
                linear     flaml
RMSE            2.2074    2.0958
MAE             1.5941    1.4732
R2              0.5203    0.5673
```

**Infer:** FLAML wins on all three, and the honest way to describe the win is "modest but
real": RMSE improves by 0.11 rings (5%), R² by 4.7 points. Two sanity checks matter more
than the direction. The test RMSE (2.0958) should land close to the cross-validated best
loss from Step 5 (2.1274) — it does, slightly better, which is what an unbiased holdout
looks like. If the test number were dramatically *worse* than the CV number, the search
had overfit its own validation folds and the reported best is optimistic. And the MAE
improvement (0.12) being proportionally larger than the RMSE improvement says the gain
comes from typical cases rather than from taming a few extreme errors.

Finally, put the size of the win in context: this dataset's ceiling is low because shell
measurements genuinely under-determine age. A 5% RMSE gain for one minute of unattended
compute is a good trade, but it is not the order-of-magnitude story AutoML marketing
implies — and on a dataset with more headroom the same 60 seconds would buy considerably
more.

## Step 6 — Read the search trace

`best_loss` is one number at the end. The trace is the shape of the whole search, and it
answers the question the single number cannot: *was the budget enough?*

In [ ]:
from flaml.automl.data import get_output_from_log

time_h, loss_h, config_h, _, _ = get_output_from_log(
    filename="abalone_flaml.log", time_budget=60
)
for t, l, c in zip(time_h, loss_h, config_h):
    learner = c["Current Learner"]
    n_est = c["Current Hyper-parameters"].get("n_estimators")
    print(f"{t:6.1f}s   rmse={l:.4f}   {learner:<11}n_estimators={n_est}")

**Observe:** the improvement-only history — roughly

```
   0.6s   rmse=2.2695   lgbm       n_estimators=4
   1.2s   rmse=2.2413   xgboost    n_estimators=4
   3.9s   rmse=2.1841   lgbm       n_estimators=11
  11.8s   rmse=2.1508   lgbm       n_estimators=29
  38.2s   rmse=2.1274   lgbm       n_estimators=63
```

**Infer:** two patterns. The **error curve flattens** — 0.05 gained in the first four
seconds, 0.02 in the next thirty-four — so the marginal value of more budget is already
small and a 10× longer run would be mostly wasted. And `n_estimators` climbs monotonically
(4 → 11 → 29 → 63): that is FLAML's cost-aware strategy visible in the data, starting with
trials that cost almost nothing and only escalating to expensive ones once cheap ones stop
paying. This is why a 60-second budget is not simply "one random configuration".

The diagnostic value is in the *contrary* case: a trace whose last improvement lands at
58s out of 60 means the search was still descending when the clock stopped, and you should
re-run with a larger budget before trusting the model. Reading only `best_loss` cannot
distinguish "converged" from "interrupted", and those call for opposite decisions.

## Where this sits against Session 4

| | Session 4 — Vertex AI AutoML | Session 25 — FLAML |
|---|---|---|
| Runs on | Google's managed infrastructure | your laptop |
| Costs | node-hours, billed | nothing |
| Needs | GCP project, billing, IAM, a bucket | `pip install` |
| Budget unit | `budget_milli_node_hours=1000` | `time_budget=60` (seconds) |
| Wall clock, small data | ~1 hour minimum | as long as you say |
| Search space | includes neural architectures | gradient boosting + sklearn estimators |
| Output | a model resource + managed endpoint | a picklable Python estimator |
| Serving | autoscaling endpoint, built in | you write it — Step 10, or Session 23 |
| Explainability | hosted feature attributions | whatever you run locally (Session 22) |
| Reproducible offline | no | yes |

The pattern to internalize is that the *AutoML* part is nearly identical — the difference
is everything around it. Vertex AI bundles search with hosting, identity, and monitoring;
FLAML gives you the search and leaves the rest to you. On tabular data of this size, the
model quality is comparable, which makes the choice one about infrastructure and budget
rather than about accuracy.

## Step 7 — Export the winning model

Persist the estimator *and* the input contract together. A pickle alone does not record
what column order it expects or what the categorical levels were, and reconstructing that
from memory three weeks later is how a serving bug gets born.

In [ ]:
import joblib, datetime, os

bundle = {
    "model": automl.model.estimator,
    "features": X_train.columns.tolist(),
    "sex_categories": X_train["Sex"].cat.categories.tolist(),
    "estimator": automl.best_estimator,
    "config": automl.best_config,
    "metrics": {"rmse": round(float(np.sqrt(mean_squared_error(y_test, pred))), 4),
                "r2": round(float(r2_score(y_test, pred)), 4)},
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
}
joblib.dump(bundle, "abalone_flaml.joblib")
print(f"saved {round(os.path.getsize('abalone_flaml.joblib')/1024, 1)} KiB")
print({k: v for k, v in bundle.items() if k != "model"})

**Observe:** `saved 118.4 KiB`, then the metadata dict with
`'features': ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight',
'Viscera_weight', 'Shell_weight']`, `'sex_categories': ['F', 'I', 'M']`, and
`'estimator': 'lgbm'`.
**Infer:** the file is ~118 KiB because the winning model is 63 small trees — an artifact
you can commit, attach to a release, or bake into a container without thinking about size.
Note `sex_categories` is stored explicitly and its order is `['F', 'I', 'M']`, alphabetical
rather than the order the values appear in the data. LightGBM encodes categories by
*position*, so a service that rebuilds the category dtype in a different order will feed
the model an `F` where it expects an `M` and return confidently wrong numbers with no
error anywhere. Persisting the level order is the fix, and Step 8's `app.py` uses it.

We save `automl.model.estimator` rather than the `AutoML` object itself so the served
artifact does not depend on FLAML being installed at inference time — only LightGBM.

## Step 8 — Serve it behind a FastAPI endpoint

This is the piece Vertex AI provided for free in Session 4 and that you now write
yourself. It is short, and it is the same shape as Session 7's API work — the difference is
only that the model inside came from a search rather than from a hand-picked estimator.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

BUNDLE = joblib.load("abalone_flaml.joblib")
MODEL, FEATURES = BUNDLE["model"], BUNDLE["features"]
SEX_CATS = BUNDLE["sex_categories"]

app = FastAPI(title="Abalone age estimator", version=BUNDLE["trained_at"])


class Abalone(BaseModel):
    Sex: str = Field(..., pattern="^[MFI]$")
    Length: float
    Diameter: float
    Height: float
    Whole_weight: float
    Shucked_weight: float
    Viscera_weight: float
    Shell_weight: float


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "estimator": BUNDLE["estimator"], "metrics": BUNDLE["metrics"]}


@app.post("/predict")
def predict(item: Abalone) -> dict:
    row = pd.DataFrame([item.model_dump()])[FEATURES]
    row["Sex"] = pd.Categorical(row["Sex"], categories=SEX_CATS)
    rings = float(MODEL.predict(row)[0])
    return {
        "predicted_rings": round(rings, 2),
        "estimated_age_years": round(rings + 1.5, 2),
        "model": BUNDLE["estimator"],
        "trained_at": BUNDLE["trained_at"],
    }

**Observe:** `Writing app.py`, and the two lines doing the real work:
`pd.DataFrame([...])[FEATURES]` and the `pd.Categorical(..., categories=SEX_CATS)` cast.
**Infer:** those two lines are the entire serving contract, and both exist to defend
against silent wrongness rather than against crashes. Indexing by `FEATURES` forces the
saved column order regardless of the order keys arrive in the JSON body — a mismatch there
does not raise, it just feeds `Diameter` into the slot the model learned as `Length`.
Rebuilding the categorical with the *saved* level list does the same job for `Sex`, per
Step 7's note.

The `pattern="^[MFI]$"` on the Pydantic field means a request with `"Sex": "male"` gets a
`422` with a readable message instead of reaching the model as an unknown category that
LightGBM silently treats as missing. And `/health` returning the training metrics means a
load balancer's health check doubles as "which model is this, and how good was it" —
cheap, and it settles arguments during an incident.

In [ ]:
!uvicorn app:app --host 127.0.0.1 --port 8000

**Observe:** the startup lines —
`INFO:     Started server process [48219]`, `INFO:     Waiting for application startup.`,
`INFO:     Application startup complete.`,
`INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)`.
**Infer:** `joblib.load` runs at *import* time, so reaching "Application startup complete"
proves the artifact from Step 7 loaded and unpickled successfully in this environment —
model loading is the startup check, and a version mismatch in LightGBM would surface here
as an unpickling traceback rather than on the first request. That property is worth
preserving deliberately: loading lazily inside the handler moves the failure from
deployment time to a user's request, which is strictly worse. Run this in a terminal (it
blocks) and use the next cell from the notebook, or append `&` to background it.

In [ ]:
import requests

print(requests.get("http://127.0.0.1:8000/health", timeout=5).json())

specimen = {
    "Sex": "M", "Length": 0.455, "Diameter": 0.365, "Height": 0.095,
    "Whole_weight": 0.514, "Shucked_weight": 0.2245,
    "Viscera_weight": 0.101, "Shell_weight": 0.15,
}
resp = requests.post("http://127.0.0.1:8000/predict", json=specimen, timeout=5)
print(resp.status_code, resp.json())

bad = dict(specimen, Sex="male")
print(requests.post("http://127.0.0.1:8000/predict", json=bad, timeout=5).status_code)

**Observe:** the health payload
(`{'status': 'ok', 'estimator': 'lgbm', 'metrics': {'rmse': 2.0958, 'r2': 0.5673}}`), then
`200 {'predicted_rings': 10.37, 'estimated_age_years': 11.87, 'model': 'lgbm',
'trained_at': '2026-08-25T11:04:18+00:00'}`, and finally `422` for the malformed request.
**Infer:** this specimen is the first row of the Abalone dataset, whose true value is 15
rings — so the endpoint is off by about 4.6 years on this particular animal. That is not a
bug and it is important not to read it as one: with test RMSE 2.10, individual errors of
this size are expected in the tail, and it is a useful antidote to judging a regression
model by one hand-picked example. Judge it by Step 5's aggregate table; use single
requests to verify the *plumbing*, which is what they can actually establish.

The `422` on `Sex: "male"` confirms validation is doing its job at the edge rather than
letting a bad value reach the model. And an endpoint that echoes its own `trained_at` and
metrics means you can tell which model answered without consulting a deployment log.

## What to try next

* Re-run Step 4 with `time_budget=600` and compare `best_loss` against the 60-second run.
  On this dataset the improvement should be small — confirming Step 6's reading of the
  trace — and confirming it yourself is how you learn to trust the trace instead of always
  reaching for a bigger budget.
* Pass `estimator_list=["lgbm"]` and then `["rf", "extra_tree"]` in separate runs to see
  how much of the win came from the *estimator family* versus the hyperparameter search.
  Useful for deciding whether the `[automl]` extra dependencies are worth carrying.
* Package this model with **Session 23**'s BentoML workflow instead of the hand-written
  FastAPI app here. Same model, two serving paths, side by side — the clearest way to feel
  what BentoML automates.
* Run the same dataset through **Session 4** (Vertex AI) or **Session 9** (SageMaker
  Autopilot) and compare not just the metrics but the wall-clock time and the dollar cost.
  That comparison, on a dataset you already understand, is what makes the cloud-versus-local
  decision concrete rather than ideological.
* Put **Session 24**'s quality gate in front of this: retrain with FLAML on every PR and
  block the merge unless the new search beats the deployed model. AutoML plus a gate is a
  genuinely automated model-improvement loop.
* Explain the winning LightGBM model with **Session 22**'s SHAP pipeline — an AutoML model
  you cannot interpret is harder to defend to a domain expert than a linear model you can,
  and on this dataset the shell-weight interaction is worth looking at.